# Fine-Tuning — Toti Cakery Tool-Calling (LoRA, 2 model Qwen3)

Melatih LoRA pada **dua kandidat keluarga Qwen3** (arch standar — stabil di T4)
dengan dataset & konfigurasi identik, lalu mengadu hasilnya pada split `test`:

| Tag | Base model (HF) | Padanan Ollama |
|---|---|---|
| `toti-qwen3-17b` | `unsloth/Qwen3-1.7B` | `qwen3:1.7b` |
| `toti-qwen3-06b` | `unsloth/Qwen3-0.6B` | `qwen3:0.6b` |

`qwen3.5:0.8b` sengaja TIDAK diikutkan: fine-tuning-nya butuh GPU ber-bf16
(dokumen resmi unsloth; di T4 terbukti LoRA tidak belajar — lihat notebook
lengkap `finetune_toti_qwen.ipynb` bila ingin menjalankannya di L4/A100).

Notebook ini ditulis dari template resmi Unsloth + `finetune/INSTRUKSI_FINETUNING.md`.
Jalankan di Colab **T4 GPU**. Estimasi total ±1.5–2 jam. Urutan: training +
tabel perbandingan dulu, export GGUF paling akhir. **Verdict resmi tetap
harness lokal** (`finetune/eval_tool_calling.py`) setelah GGUF di-deploy.

### Installation

(Cell instalasi diambil apa adanya dari template resmi Unsloth.)

In [ ]:
%%capture
import os, importlib.util
!pip install --upgrade -qqq uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
    except: _numpy = "numpy"; _pil = "pillow"
    !uv pip install -qqq \
        "torch==2.8.0" "triton>=3.3.0" {_numpy} {_pil} torchvision bitsandbytes xformers==0.0.32.post2 \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth"
    !uv pip install -qqq --no-deps "torchcodec==0.7.0"
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth
!uv pip install --upgrade --no-deps "tokenizers>=0.22.0,<=0.23.0" trl==0.22.2 unsloth unsloth_zoo
!uv pip install transformers==5.2.0
# causal_conv1d is supported only on torch==2.8.0. If you have newer torch versions, please wait 10 minutes!
!uv pip install --no-build-isolation flash-linear-attention causal_conv1d==1.6.0
import torch
if torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8:
    !uv pip install --no-deps "apache-tvm-ffi==0.1.9" "tilelang==0.1.8"
else:
    os.environ["FLA_TILELANG"] = "0"
!uv pip install --no-deps --upgrade "torchao>=0.16.0"

<a name="Data"></a>
### Data Prep — dataset Toti Cakery tool-calling

Dataset: **`LasagnaS/toti-cakery-toolcall`** (public, identik dengan
`finetune/data/` repo). Dimuat MENTAH (kolom `messages` + `tools_json`) — render
ke kolom `text` dilakukan **per model di dalam pipeline**, karena chat template
tiap keluarga beda (Qwen3.5 → tool call XML `<function=...>`, Qwen3 → JSON
`{"name":...}`).

⚠️ Split `test` hanya untuk evaluasi — tidak pernah masuk training/val.

In [ ]:
from datasets import load_dataset
from huggingface_hub import hf_hub_download

# CATATAN: load_dataset("LasagnaS/toti-cakery-toolcall") langsung bisa gagal di
# datasets versi lama — "Feature type 'Json' not found" — karena metadata yang
# diekspor Hub dibuat dengan datasets versi baru. Unduh jsonl-nya langsung lalu
# muat via builder "json" (tahan semua versi):
files = {split: hf_hub_download("LasagnaS/toti-cakery-toolcall",
                                f"data/{split}.jsonl", repo_type = "dataset")
         for split in ("train", "validation", "test")}
ds = load_dataset("json", data_files = files)

print(ds)
# v3 (Jul 2026): train 985 / val 99 (v3: +T5/T11/T12); test 100 dibekukan dari v1
assert len(ds["train"]) == 985,      f"train harus 985 baris (v3), dapat {len(ds['train'])}"
assert len(ds["validation"]) == 99,  f"validation harus 99 baris (v3), dapat {len(ds['validation'])}"
assert len(ds["test"]) == 100,       f"test harus 100 baris, dapat {len(ds['test'])}"


### Util: render per model, sanity check, dan eval harness in-notebook

- `render_ds(tk)` — render `messages`+`tools` → kolom `text` memakai chat
  template tokenizer yang diberikan (cell konversi dari `finetune/README.md`,
  di-parameterkan per model). `arguments` (JSON string) di-parse jadi dict
  SEBELUM `apply_chat_template` — template diam-diam membuang argumen non-dict.
- `sanity_check(dsm)` — WAJIB lolos sebelum training: round-trip render→parse
  vs gold, format-agnostic (jalan utk XML Qwen3.5 maupun JSON Qwen3).
- `run_eval(tag, m, tk)` — replay split `test`, definisi metrik **persis**
  `finetune/eval_tool_calling.py`. Kondisi meniru produksi: **thinking NYALA**
  (terverifikasi via `/api/chat` bahwa Ollama default thinking-on), sampling
  produksi `temperature 0.7 / top_p 0.8` (`app/core/config.py`), cap 768 token
  (= `num_predict` harness; termasuk token thinking — thinking terpotong =
  "tanpa call", sama seperti di jalur produksi). Diagnostik `truncated_rows` /
  `think_unclosed_rows` / `avg_gen_tokens` memisahkan "salah pilih tool" dari
  "kehabisan jatah token".

In [ ]:
import json
import random
import re
import time
import torch
from collections import defaultdict
from tqdm.auto import tqdm

FUNC_RE  = re.compile(r"<function=([^>\n]+)>(.*?)</function>", re.S)
PARAM_RE = re.compile(r"<parameter=([^>\n]+)>\n?(.*?)\n?</parameter>", re.S)

def render_ds(tk):
    """Render dataset mentah → kolom `text` dengan chat template model ybs."""
    def to_text(ex):
        msgs = ex["messages"]
        for m in msgs:
            for tc in (m.get("tool_calls") or []):
                if isinstance(tc["function"]["arguments"], str):
                    tc["function"]["arguments"] = json.loads(tc["function"]["arguments"])
        return {"text": tk.apply_chat_template(
            msgs, tools = json.loads(ex["tools_json"]),
            tokenize = False, add_generation_prompt = False)}
    return ds.map(to_text)

def strip_nones(o):
    if isinstance(o, dict):
        return {k: strip_nones(v) for k, v in o.items() if v is not None}
    if isinstance(o, list):
        return [strip_nones(v) for v in o]
    return o

def gold_of(row):
    """Tool emas (nama, args) dari turn assistant final — (None, None) utk baris non-tool."""
    final = row["messages"][-1]
    if final.get("tool_calls"):
        fn = final["tool_calls"][0]["function"]
        raw = fn["arguments"]
        return fn["name"], (json.loads(raw) if isinstance(raw, str) else strip_nones(raw))
    return None, None

def canon_args(obj):
    """Normalisasi utk exact-match — persis finetune/eval_tool_calling.py."""
    if isinstance(obj, dict):
        return {k: canon_args(v) for k, v in sorted(obj.items())}
    if isinstance(obj, list):
        return [canon_args(v) for v in obj]
    if isinstance(obj, str) and obj.isdigit():
        return obj  # string angka tetap string; qty salah tipe dihitung salah
    return obj

def parse_first_tool_call(text):
    """(nama, args, invalid) dari output mentah. Mendukung DUA format:
    - Qwen3.5 : <tool_call><function=nama><parameter=k>v</parameter></function></tool_call>
    - Qwen3   : <tool_call>{"name": "nama", "arguments": {...}}</tool_call>
    Blok thinking dibuang dulu; nilai parameter XML dibaca via json.loads bila
    valid (int/list/dict), selain itu string."""
    text = text.split("<|im_end|>")[0]
    # Dua bentuk thinking: (a) <think>...</think> lengkap di output (Qwen3),
    # (b) hanya </think> karena <think> di-prefill di prompt (Qwen3.5,
    # enable_thinking=True). Terpotong tanpa </think> = tidak sempat menjawab.
    if "</think>" in text:
        text = text.split("</think>", 1)[1]
    elif "<think>" in text:
        text = ""
    if "<tool_call>" not in text:
        return None, None, False
    body = text.split("<tool_call>", 1)[1].split("</tool_call>")[0].strip()
    m = FUNC_RE.search(body)
    if m:  # format XML Qwen3.5
        args = {}
        for k, v in PARAM_RE.findall(m.group(2)):
            v = v.strip()
            try:
                args[k.strip()] = json.loads(v)
            except Exception:
                args[k.strip()] = v
        return m.group(1).strip(), args, False
    try:   # format JSON Qwen3
        obj = json.loads(body)
        return obj["name"], (obj.get("arguments") or {}), False
    except Exception:
        return None, None, True  # mulai <tool_call> tapi tidak bisa diparse

def sanity_check(dsm):
    """WAJIB sebelum training: render→parse round-trip harus cocok dengan gold."""
    random.seed(42)
    tool_idx  = [i for i, m in enumerate(dsm["train"]["messages"]) if m[-1].get("tool_calls")]
    plain_idx = [i for i, m in enumerate(dsm["train"]["messages"]) if not m[-1].get("tool_calls")]
    for i in random.sample(tool_idx, 25):
        row = dsm["train"][i]
        gname, gargs = gold_of(row)
        out = row["text"].rsplit("<|im_start|>assistant\n", 1)[1]
        name, args, invalid = parse_first_tool_call(out + "<|im_end|>")
        assert name == gname and not invalid, f"baris {i}: {name!r} != {gname!r} (invalid={invalid})"
        assert canon_args(args) == canon_args(gargs), f"baris {i}: args {args} != {gargs}"
    for i in random.sample(plain_idx, 10):
        out = dsm["train"][i]["text"].rsplit("<|im_start|>assistant\n", 1)[1]
        name, _a, inv = parse_first_tool_call(out + "<|im_end|>")
        assert name is None and not inv, f"baris {i}: false tool di baris non-tool"
    print(f"sanity render OK ({len(tool_idx)} baris tool / {len(plain_idx)} non-tool)")
    print("===== CONTOH AKHIR RENDER BARIS TOOL-CALL =====")
    print(dsm["train"][tool_idx[0]]["text"][-500:])

def run_eval(tag, m, tk, batch_size = 16, max_new_tokens = 768, chat_kwargs = None):
    """Replay seluruh split test; definisi metrik persis eval_tool_calling.py.
    max_new_tokens 768 = cap num_predict harness. THINKING default NYALA
    (enable_thinking=True) — paritas dgn default Ollama di produksi/harness."""
    from unsloth import FastLanguageModel
    if chat_kwargs is None:
        chat_kwargs = {"enable_thinking": True}
    FastLanguageModel.for_inference(m)
    # for_inference di arch qwen3_5 dicurigai menonaktifkan adapter LoRA →
    # pastikan adapter AKTIF saat eval (no-op kalau sudah aktif / bukan PEFT)
    for _obj in (m, getattr(m, "base_model", None)):
        if _obj is not None and hasattr(_obj, "enable_adapter_layers"):
            try:
                _obj.enable_adapter_layers()
            except Exception:
                pass
    for _mod in m.modules():  # buang cache rope_deltas basi (lihat catatan training)
        if hasattr(_mod, "rope_deltas"):
            _mod.rope_deltas = None
    torch.manual_seed(42)  # sampling produksi, seed tetap → antar-run sebanding
    t0 = time.time()

    prompts, golds, rtypes = [], [], []
    for row in ds["test"]:
        msgs = [dict(x) for x in row["messages"][:-1]]
        prompts.append(tk.apply_chat_template(
            msgs, tools = json.loads(row["tools_json"]),
            tokenize = False, add_generation_prompt = True, **chat_kwargs))
        golds.append(gold_of(row))
        rtypes.append(row["meta"]["type"])

    # FastLanguageModel utk Qwen3.5 mengembalikan Processor multimodal — argumen
    # posisi pertamanya `images`, jadi encode/decode HARUS lewat tokenizer teks
    # di dalamnya (kalau tidak, prompt dicoba dibuka sebagai gambar).
    tok = getattr(tk, "tokenizer", tk)
    old_side, tok.padding_side = tok.padding_side, "left"
    outputs = []
    for b in tqdm(range(0, len(prompts), batch_size), desc = tag):
        enc = tok(prompts[b:b + batch_size], return_tensors = "pt",
                  padding = True).to("cuda")
        # sampling = config PRODUKSI chatbot (app/core/config.py: temperature 0.7,
        # top_p 0.8 — juga dipakai harness lokal), bukan default Modelfile
        out = m.generate(**enc, max_new_tokens = max_new_tokens, do_sample = True,
                         temperature = 0.7, top_p = 0.8, top_k = 20,
                         use_cache = True, pad_token_id = tok.pad_token_id)
        outputs += tok.batch_decode(out[:, enc["input_ids"].shape[1]:])
    tok.padding_side = old_side

    per_type = defaultdict(lambda: {"n": 0, "sel": 0, "param": 0, "irrelevant_ok": 0,
                                    "false_tool": 0, "invalid": 0})
    for (gname, gargs), rtype, raw in zip(golds, rtypes, outputs):
        name, args, invalid = parse_first_tool_call(raw)
        st = per_type[rtype]
        st["n"] += 1
        if gname is None:            # baris non-tool
            if name is not None:
                st["false_tool"] += 1
            else:                    # percobaan invalid tanpa call ikut sini — mirror harness
                st["irrelevant_ok"] += 1
        else:                        # baris tool
            if invalid and name is None:
                st["invalid"] += 1
            if name == gname:
                st["sel"] += 1
                if canon_args(args) == canon_args(gargs):
                    st["param"] += 1

    tool_types = [t for t in per_type if t.startswith("T")]
    non_types  = [t for t in per_type if t.startswith("N")]
    tool_n = sum(per_type[t]["n"] for t in tool_types)
    non_n  = sum(per_type[t]["n"] for t in non_types)
    agg = {
        "tag": tag,
        "function_selection_acc": round(sum(per_type[t]["sel"] for t in tool_types) / max(1, tool_n), 3),
        "param_exact_acc":        round(sum(per_type[t]["param"] for t in tool_types) / max(1, tool_n), 3),
        "invalid_call_rate":      round(sum(per_type[t]["invalid"] for t in tool_types) / max(1, tool_n), 3),
        "irrelevance_acc":        round(sum(per_type[t]["irrelevant_ok"] for t in non_types) / max(1, non_n), 3),
        "false_tool_rate":        round(sum(per_type[t]["false_tool"] for t in non_types) / max(1, non_n), 3),
        "per_type": {t: dict(per_type[t]) for t in sorted(per_type)},
        "seconds": round(time.time() - t0, 1),
        # Diagnostik: memisahkan "salah pilih tool" dari "kehabisan jatah token".
        "truncated_rows":      sum(1 for o in outputs if "<|im_end|>" not in o),
        "think_unclosed_rows": sum(1 for o in outputs
                                   if "</think>" not in o.split("<|im_end|>")[0]),
        "avg_gen_tokens":      round(sum(len(tok(o).input_ids) for o in outputs)
                                     / max(1, len(outputs)), 1),
    }
    print(json.dumps({k: v for k, v in agg.items() if k != "per_type"}, indent = 2))
    print(f"⏱️ {agg['seconds']}s untuk {len(prompts)} baris "
          f"({agg['seconds'] / max(1, len(prompts)):.1f}s/baris) | "
          f"terpotong cap: {agg['truncated_rows']} | thinking tak selesai: "
          f"{agg['think_unclosed_rows']} | rata-rata {agg['avg_gen_tokens']} token/baris")
    return agg

In [ ]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

<a name="Pipeline"></a>
### Pipeline per model: load → sanity → baseline → LoRA train → eval → export

Konfigurasi **identik** untuk ketiga model (perbandingan adil, sesuai INSTRUKSI):
LoRA r=16, α=16, dropout=0, semua proyeksi linear, seed 42; lr 2e-4 linear,
warmup 10 step, 2 epoch, batch efektif 8, eval val-loss tiap 25 step;
**aturan berhenti**: `EarlyStoppingCallback(patience=2)` + `load_best_model_at_end`
(val-loss tak membaik 2 evaluasi berturut-turut → stop, pakai checkpoint terbaik).

Catatan teknis yang sudah tertanam (jangan diubah):
- `processing_class` = tokenizer TEKS internal, bukan Processor multimodal —
  kalau Processor, TRL membangun collator ber-`image_processor` dan
  `train_on_responses_only` menolak (vision collator).
- Reset `model.rope_deltas` sebelum `trainer.train()` — cache sisa `generate()`
  baseline membuat forward training crash di transformers==5.2.0
  ("size of tensor a (2) must match ... (0)"; diperbaiki di >= 5.5).
- Masking `train_on_responses_only` marker Qwen: loss hanya pada respons
  assistant; diverifikasi dengan decode label + assert system prompt tak ikut.
- Setelah selesai: hanya LoRA → `lora_<tag>/` (cepat), lalu model dibongkar
  dari VRAM. **Export GGUF dipisah** ke seksi setelah tabel perbandingan —
  training + hasil semua model diprioritaskan dulu.

In [ ]:
import gc
import glob
import os
import matplotlib.pyplot as plt
from unsloth import FastLanguageModel
from unsloth.chat_templates import train_on_responses_only
from trl import SFTTrainer, SFTConfig
from transformers import EarlyStoppingCallback

SMOKE = [
    ("sapaan — JANGAN panggil tool",          "halo kak, selamat siang!"),
    ("ambigu N5 — harus bertanya balik",      "mau pesan kue dong"),
    ("adversarial N6 — BUKAN cancel_order",   "cara batalin pesanan gimana sih kak?"),
    ("out-of-scope — tolak dengan sopan",     "kak tau resep rendang yang enak nggak?"),
    ("tool-call — add_to_cart",               "pesan brownies coklat 2 kotak ya"),
]

def run_smoke(m, tk, tag):
    """Cek kualitatif singkat (dibaca mata, bukan verdict) — thinking nyala."""
    sys_msg = {"role": "system", "content": ds["test"][0]["messages"][0]["content"]}
    smoke_tools = json.loads(ds["test"][0]["tools_json"])
    tok = getattr(tk, "tokenizer", tk)
    for label, q in SMOKE:
        prompt = tk.apply_chat_template(
            [sys_msg, {"role": "user", "content": q}], tools = smoke_tools,
            tokenize = False, add_generation_prompt = True, enable_thinking = True)
        enc = tok(prompt, return_tensors = "pt").to("cuda")
        t0 = time.time()
        out = m.generate(**enc, max_new_tokens = 512, do_sample = True,
                         temperature = 0.7, top_p = 0.8, top_k = 20,
                         use_cache = True, pad_token_id = tok.pad_token_id)
        dt = time.time() - t0
        raw = tok.decode(out[0, enc["input_ids"].shape[1]:]).split("<|im_end|>")[0]
        think, _, ans = raw.rpartition("</think>")
        print(f"\n===== [{tag}] {label}  (⏱️ {dt:.1f}s)\nUSER : {q}")
        if think.strip():
            print(f"THINK: {think.strip()[:200]}")
        print(f"BOT  : {ans.strip()[:400]}")

def finetune_and_eval(hf_name, tag):
    print(f"\n{'#' * 70}\n# {tag}  ←  {hf_name}\n{'#' * 70}")

    # ---- 1. Load (jalur teks; max_seq_length 4096 — prompt terpanjang ±1.9k tok)
    # Qwen3.5 di T4 (unsloth #4970): checkpoint bf16 + T4 tanpa bf16 native.
    # Crash dtype selalu terjadi di RECOMPUTE backward gradient-checkpointing
    # unsloth (forward-nya sendiri sukses) → utk Qwen3.5 matikan gradient
    # checkpointing (backward memakai graf forward asli, mismatch mustahil)
    # dan kompensasi VRAM via batch 1 × accum 8. Keluarga Qwen3 standar tetap
    # konfigurasi normal.
    is_qwen35 = "Qwen3.5" in hf_name
    t4 = globals().get("T4_MODE", not torch.cuda.is_bf16_supported())
    hard_mode = is_qwen35 and t4  # qwen3.5 di GPU tanpa bf16 — jalur bermasalah
    gc_mode = False if hard_mode else "unsloth"
    model, tokenizer = FastLanguageModel.from_pretrained(
        hf_name,
        max_seq_length = 4096,
        dtype = torch.float32 if hard_mode else None,  # GPU bf16: biarkan auto (bf16)
        load_in_4bit = False,    # QLoRA tidak disarankan utk Qwen3.5 (dok resmi)
        use_gradient_checkpointing = gc_mode,
    )
    tok = getattr(tokenizer, "tokenizer", tokenizer)  # tokenizer teks internal

    # ---- 2. Render dataset dgn template model INI + sanity (WAJIB lolos)
    dsm = render_ds(tokenizer)
    sanity_check(dsm)

    # ---- 3. LoRA (INSTRUKSI §2)
    model = FastLanguageModel.get_peft_model(
        model,
        r = 16,
        target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                          "gate_proj", "up_proj", "down_proj"],
        lora_alpha = 16,
        lora_dropout = 0,
        bias = "none",
        use_gradient_checkpointing = gc_mode,
        random_state = 42,
        use_rslora = False,
        loftq_config = None,
    )

    def lora_b_signature():
        """Σ|lora_B| — nol saat init; harus > 0 setelah training yang benar."""
        return sum(p.detach().float().abs().sum().item()
                   for n, p in model.named_parameters() if "lora_B" in n)

    print(f"Σ|lora_B| sebelum training: {lora_b_signature():.6f} (harus 0 — init)")

    # ---- 4. Baseline SEBELUM training (LoRA baru = identitas, aman)
    baseline = run_eval(f"BASELINE {tag}", model, tokenizer)

    # ---- 5. Trainer + masking respons-only
    FastLanguageModel.for_training(model)
    trainer = SFTTrainer(
        model = model,
        processing_class = tok,   # BUKAN Processor multimodal (lihat markdown)
        train_dataset = dsm["train"],
        eval_dataset = dsm["validation"],
        # patience 4 (bukan 2): aturan INSTRUKSI = berhenti saat val-loss NAIK
        # berkelanjutan; EarlyStoppingCallback menghitung "tidak membaik" yang
        # lebih galak — patience 2 menyetop di step 75 saat kurva cuma datar,
        # menyisakan checkpoint ¼-epoch yang kurang latih. load_best_model_at_end
        # tetap jadi pengaman overfit yang sebenarnya.
        callbacks = [EarlyStoppingCallback(early_stopping_patience = 4)],
        args = SFTConfig(
            dataset_text_field = "text",
            max_length = 4096,
            # Batch efektif 8 utk semua (paritas hyperparameter). Non-qwen3.5
            # dikomposisi 2×4: 8×1 bikin EVAL step OOM di T4 (eval memakai jalur
            # non-fused → materialisasi logit fp32 batch penuh ~7GB, tak muat
            # di atas train batch 8) DAN tak lebih cepat (compute-bound, 230 step
            # ~40 mnt sama saja). qwen3.5-di-T4 tetap 1×8.
            per_device_train_batch_size = 1 if hard_mode else 2,
            gradient_accumulation_steps = 8 if hard_mode else 4,
            num_train_epochs = 2,
            warmup_steps = 10,
            learning_rate = 2e-4,
            lr_scheduler_type = "linear",
            logging_steps = 5,
            optim = "adamw_8bit",
            weight_decay = 0.001,
            seed = 42,
            output_dir = f"outputs_{tag}",
            report_to = "none",
            per_device_eval_batch_size = 2 if hard_mode else 4,  # kecil: eval-loss tak terpengaruh, hindari OOM logit
            eval_strategy = "steps",
            eval_steps = 25,
            save_strategy = "steps",
            save_steps = 25,
            save_total_limit = 2,
            load_best_model_at_end = True,
            metric_for_best_model = "eval_loss",
            greater_is_better = False,
        ),
    )
    ct = getattr(tok, "chat_template", None) or getattr(tokenizer, "chat_template", "")
    assert "<|im_start|>user" in ct and "<|im_start|>assistant" in ct, (
        "Marker Qwen tidak ditemukan di chat_template aktual!")
    trainer = train_on_responses_only(
        trainer,
        instruction_part = "<|im_start|>user\n",
        response_part    = "<|im_start|>assistant\n",
    )

    # verifikasi masking: token ber-loss harus HANYA respons assistant
    ex = trainer.train_dataset[0]
    labels = ex["labels"]
    kept = [t for t, l in zip(ex["input_ids"], labels) if l != -100]
    assert 0 < len(kept) < len(labels), "masking gagal total"
    decoded_kept = tok.decode(kept)
    assert "Toti Cakery, sebuah toko kue" not in decoded_kept, (
        "System prompt ikut terlatih — masking SALAH, stop!")
    print(f"masking OK: {len(kept)}/{len(labels)} token dilatih; contoh: "
          f"{decoded_kept[:120]!r}")

    # ---- 6. Train (reset rope_deltas dulu — cache sisa generate() baseline)
    for _m in model.modules():
        if hasattr(_m, "rope_deltas"):
            _m.rope_deltas = None
    trainer_stats = trainer.train()
    print(f"training {tag}: {trainer_stats.metrics['train_runtime']:.0f}s "
          f"({trainer_stats.metrics['train_runtime'] / 60:.1f} menit)")

    lora_b_after = lora_b_signature()
    print(f"Σ|lora_B| setelah training: {lora_b_after:.6f}")
    if lora_b_after == 0.0:
        print("🚨 LoRA TIDAK BELAJAR SAMA SEKALI (lora_B masih nol) — hasil "
              "fine-tuned di bawah = base model. Jangan pakai GGUF-nya; "
              "laporkan ke Kevin/Claude sebelum lanjut.")

    # ---- 7. Kurva loss (cek overfitting: val naik saat train turun = jelek)
    hist = trainer.state.log_history
    tr = [(h["step"], h["loss"]) for h in hist if "loss" in h]
    ev = [(h["step"], h["eval_loss"]) for h in hist if "eval_loss" in h]
    for s, l in ev:
        print(f"  step {s:>4}  eval_loss {l:.4f}")
    fig, ax = plt.subplots(figsize = (7, 3))
    if tr: ax.plot(*zip(*tr), color = "#4477AA", lw = 2, label = "train loss")
    if ev: ax.plot(*zip(*ev), color = "#EE7733", lw = 2, marker = "o", ms = 6, label = "val loss")
    ax.set_xlabel("step"); ax.set_ylabel("loss")
    ax.grid(alpha = 0.25); ax.legend(frameon = False)
    ax.set_title(f"Train vs validation loss — {tag}", loc = "left")
    plt.tight_layout(); plt.show()

    # ---- 8. Eval SESUDAH training (kondisi identik dgn baseline) + smoke
    finetuned = run_eval(f"FINE-TUNED {tag}", model, tokenizer)
    run_smoke(model, tokenizer, tag)

    # ---- 9. Simpan LoRA saja (cepat, ±50MB). Export GGUF sengaja DIPISAH ke
    # seksi setelah tabel perbandingan — prioritas: hasil semua model dulu,
    # export hanya untuk model yang dipilih.
    model.save_pretrained(f"lora_{tag}")
    tokenizer.save_pretrained(f"lora_{tag}")
    print(f"LoRA tersimpan: lora_{tag}/ — export GGUF di seksi akhir notebook")

    # ---- 10. Bongkar dari VRAM utk model berikutnya
    del trainer, model, tokenizer, tok, dsm
    gc.collect()
    torch.cuda.empty_cache()
    return {"baseline": baseline, "finetuned": finetuned,
            "train_seconds": round(trainer_stats.metrics["train_runtime"], 1)}

### Jalankan per model — dua cell terpisah (resumable)

Kalau Colab timeout di tengah: restart runtime → jalankan ulang cell instalasi,
data, util, pipeline → lalu HANYA cell model yang belum selesai. Kalau muncul
error CUDA/VRAM aneh setelah ganti model, restart runtime juga (paling bersih).

In [ ]:
all_results = globals().get("all_results", {})
all_results["toti-qwen3-17b"] = finetune_and_eval("unsloth/Qwen3-1.7B", "toti-qwen3-17b")

In [ ]:
all_results = globals().get("all_results", {})
all_results["toti-qwen3-06b"] = finetune_and_eval("unsloth/Qwen3-0.6B", "toti-qwen3-06b")

<a name="Compare"></a>
### Tabel perbandingan akhir — 3 model, baseline → fine-tuned vs target

Kolom `harness` = baseline di jalur produksi sungguhan (Ollama CPU, mesin lokal;
`finetune/results_*.json`) sebagai jangkar realitas. Model terbaik = yang lolos
paling banyak target; seri → pilih yang lebih kecil/cepat.

In [ ]:
TARGETS = {
    "function_selection_acc": (">=", 0.80),
    "param_exact_acc":        (">=", 0.65),
    "irrelevance_acc":        (">=", 0.80),
    "false_tool_rate":        ("<=", 0.15),
    "invalid_call_rate":      ("<=", 0.05),  # "tetap ~0"
}
# Baseline jalur produksi (harness lokal, results_*.json). None = belum diukur.
HARNESS_BASELINE = {
    "toti-qwen35-08b": {"function_selection_acc": 0.417, "param_exact_acc": 0.300,
                        "irrelevance_acc": 0.575, "false_tool_rate": 0.425,
                        "invalid_call_rate": 0.000},
    "toti-qwen3-17b": {"function_selection_acc": 0.550, "param_exact_acc": 0.217,
                       "irrelevance_acc": 0.875, "false_tool_rate": 0.125,
                       "invalid_call_rate": 0.000},
    "toti-qwen3-06b": {"function_selection_acc": 0.267, "param_exact_acc": 0.050,
                       "irrelevance_acc": 0.975, "false_tool_rate": 0.025,
                       "invalid_call_rate": 0.000},
}

scoreboard = {}
for tag, res in all_results.items():
    b, f = res["baseline"], res["finetuned"]
    h = HARNESS_BASELINE.get(tag)
    print(f"\n===== {tag} =====")
    print(f"{'metrik':<24} {'harness':>8} {'base-nb':>8} {'ft-nb':>8} {'delta':>7}  target   status")
    hits = 0
    for mname, (op, tgt) in TARGETS.items():
        hv = f"{h[mname]:.3f}" if h else "-"
        hit = (f[mname] >= tgt) if op == ">=" else (f[mname] <= tgt)
        hits += hit
        print(f"{mname:<24} {hv:>8} {b[mname]:>8.3f} {f[mname]:>8.3f} "
              f"{f[mname] - b[mname]:>+7.3f}  {op} {tgt:<4}  {'✅' if hit else '❌'}")
    scoreboard[tag] = hits
    print(f"target tercapai: {hits}/{len(TARGETS)} | waktu eval ft: {f['seconds']}s | "
          f"trunc: {f['truncated_rows']} | training: {res['train_seconds']}s")
    print("per-type (baseline → fine-tuned):")
    for t in sorted(f["per_type"]):
        bb, ff = b["per_type"].get(t, {}), f["per_type"][t]
        if t.startswith("T"):
            print(f"  {t:4} n={ff['n']:3}  sel {bb.get('sel', 0):3} → {ff['sel']:3}   "
                  f"param {bb.get('param', 0):3} → {ff['param']:3}")
        else:
            print(f"  {t:4} n={ff['n']:3}  ok  {bb.get('irrelevant_ok', 0):3} → {ff['irrelevant_ok']:3}   "
                  f"false_tool {bb.get('false_tool', 0):3} → {ff['false_tool']:3}")

if scoreboard:
    best = max(scoreboard, key = lambda t: (scoreboard[t],
               all_results[t]["finetuned"]["function_selection_acc"]))
    print(f"\n🏆 Kandidat terbaik in-notebook: {best} "
          f"({scoreboard[best]}/{len(TARGETS)} target) — KONFIRMASI dengan harness "
          f"lokal setelah deploy GGUF (verdict resmi).")

<a name="Export"></a>
### Export GGUF — jalankan SETELAH tabel perbandingan

LoRA tiap model sudah di disk (`lora_<tag>/`); cell ini me-reload lalu
mengonversi ke GGUF q8_0. Default meng-export semua model Qwen3 yang selesai —
**persempit `EXPORT_TAGS` ke pemenang saja** kalau mau hemat waktu (±10-15
menit/model). `toti-qwen35-08b` default TIDAK diikutkan: konverter GGUF belum
mengenal tensor MTP (`eh_proj`) arsitektur Qwen3.5.

In [ ]:
# Cell ini MANDIRI: aman dijalankan di session baru (setelah restart) selama
# folder lora_<tag>/ masih ada di disk — cukup jalankan cell instalasi dulu.
import gc
import glob
import os
import torch
from unsloth import FastLanguageModel

# Default: semua LoRA yang ditemukan di disk (bukan dari variabel memori)
EXPORT_TAGS = [d[len("lora_"):] for d in sorted(glob.glob("lora_*")) if os.path.isdir(d)]
EXPORT_TAGS = [t for t in EXPORT_TAGS if "qwen35" not in t]  # 0.8b: konverter GGUF belum support
# EXPORT_TAGS = ["toti-qwen3-17b"]   # ← contoh: hanya pemenang
print("akan di-export:", EXPORT_TAGS or "(tidak ada lora_*/ di disk!)")

for tag in EXPORT_TAGS:
    print(f"\n===== export {tag} =====")
    m2, t2 = FastLanguageModel.from_pretrained(
        f"lora_{tag}",
        max_seq_length = 4096,
        dtype = torch.float32 if "qwen35" in tag else None,
        load_in_4bit = False,
    )
    try:
        m2.save_pretrained_gguf(f"gguf_{tag}", t2)  # default Q8_0
        for f in glob.glob(f"gguf_{tag}*/**/*.gguf", recursive = True) + glob.glob(f"gguf_{tag}*.gguf"):
            print("GGUF:", f, f"{os.path.getsize(f) / 1e9:.2f} GB")
        # Alternatif download manual (±1GB): push langsung ke HF
        # m2.push_to_hub_gguf(f"HF_USERNAME/{tag}-gguf", t2, token = "HF_TOKEN")
    except Exception as exc:
        print(f"⚠️ Export GGUF gagal {tag}: {exc} — LoRA tetap aman di lora_{tag}/")
    del m2, t2
    gc.collect()
    torch.cuda.empty_cache()

### Push GGUF ke HuggingFace — jalankan SETELAH cell export

Meng-upload `.gguf` yang baru dibuat ke repo model HF (default **privat**).
Nama file di Hub sudah = nama produksi yang dipakai Modelfile di `finetune/`
(`toti-qwen-1.7b.Q8_0.gguf`, `toti-qwen-0.6b.Q8_0.gguf`) → di mesin lokal
tinggal `huggingface-cli download` lalu `ollama create`, tanpa rename.

Token: dipakai token login HF yang sama seperti saat load model (`get_token()`);
kalau kosong, cell akan meminta lewat prompt. Butuh token **write**.


In [ ]:
# =========================================================================
# Push GGUF -> HuggingFace Hub (jalankan SETELAH cell export GGUF di atas)
# =========================================================================
import glob, os, getpass
from huggingface_hub import HfApi, create_repo, get_token

HF_REPO    = "LasagnaS/toti-qwen-gguf"   # repo model tujuan
HF_PRIVATE = True                        # repo privat (ubah ke False bila publik)

# Nama file akhir di Hub per tag = persis nama yang diharapkan Modelfile lokal.
HUB_NAME = {
    "toti-qwen3-17b": "toti-qwen-1.7b.Q8_0.gguf",
    "toti-qwen3-06b": "toti-qwen-0.6b.Q8_0.gguf",
}

token = os.environ.get("HF_TOKEN") or get_token() or getpass.getpass("HF token (write): ")
api = HfApi(token=token)
create_repo(HF_REPO, repo_type="model", private=HF_PRIVATE, exist_ok=True, token=token)

pushed = []
for tag, dst in HUB_NAME.items():
    hits = sorted(glob.glob(f"gguf_{tag}*/**/*.gguf", recursive=True) +
                  glob.glob(f"gguf_{tag}*.gguf"))
    if not hits:
        print(f"skip {tag}: tak ada .gguf (export dulu cell di atas)")
        continue
    src = hits[0]
    print(f"upload {tag}: {src} -> {HF_REPO}/{dst} ({os.path.getsize(src)/1e9:.2f} GB) ...")
    api.upload_file(path_or_fileobj=src, path_in_repo=dst,
                    repo_id=HF_REPO, repo_type="model")
    pushed.append(dst)

print("\nselesai. ter-push:", pushed or "(tidak ada)")
print(f"  https://huggingface.co/{HF_REPO}/tree/main")
# Download di mesin lokal Kevin:
#   huggingface-cli download {HF_REPO} toti-qwen-1.7b.Q8_0.gguf --local-dir finetune/
#   huggingface-cli download {HF_REPO} toti-qwen-0.6b.Q8_0.gguf --local-dir finetune/


### Deploy ke Ollama + verdict resmi (di mesin lokal Kevin)

Download GGUF tiap model (atau `push_to_hub_gguf`), taruh di `finetune/`, lalu:

| Tag | Rename GGUF menjadi | Perintah create |
|---|---|---|
| `toti-qwen3-17b` | `toti-qwen-1.7b.Q8_0.gguf` | `ollama create toti-qwen-1.7b -f Modelfile.qwen3-1.7b` |
| `toti-qwen3-06b` | `toti-qwen-0.6b.Q8_0.gguf` | `ollama create toti-qwen-0.6b -f Modelfile.qwen3-0.6b` |

Modelfile per keluarga SUDAH ada di `finetune/` dengan template asli base
model masing-masing (qwen3.5 pakai `RENDERER/PARSER qwen3.5`; qwen3 pakai
Go-TEMPLATE penuh). **Template beda = penyebab #1 "kok jadi bodoh setelah
export"** — jangan tulis Modelfile sendiri.

```bash
cd /home/kevin/clcode/chatbot
# verdict resmi per model (±50 mnt-2.5 jam per model di CPU; jalankan background):
chatbot-service/.venv/bin/python finetune/eval_tool_calling.py --model toti-qwen-1.7b
chatbot-service/.venv/bin/python finetune/eval_tool_calling.py --model toti-qwen-0.6b
# bandingkan dgn finetune/results_qwen3.5_0.8b.json, results_qwen3_1.7b.json,
# results_qwen3_0.6b.json (baseline produksi)
```

Model pemenang → `LLM_MODEL=<nama>` di `.env`, smoke `python -m scripts.chat_cli`,
dan `pytest chatbot-service/tests/` harus tetap hijau. Gagal target → diagnosis
INSTRUKSI §4 (sanity render → masking → 3 epoch / lr 1e-4 → per-type).